# 04 - Behavioral Segmentation

## Objective
Add a behavioral intelligence layer on top of churn labels by segmenting 
users based on activity level and purchase behavior.

## Segmentation Logic
- Activity Level: High (>= median events) vs Low (< median events)
- Median total events threshold: 75
- Segment created by combining activity level + churn label

## Quiet Loyal Users Rule
Low Activity + Active + purchase frequency >= median frequency
These users visit infrequently but convert consistently when they do.

## Final Segments
- High Activity — Active:                  499 (49.9%)
- Low Activity — Active:                   243 (24.3%)
- Low Activity — Retained (Quiet Loyal):   186 (18.6%)
- High Activity — Potential Churn:          34 (3.4%)
- Low Activity — Potential Churn:           32 (3.2%)
- Low Activity — Churned:                    3 (0.3%)
- High Activity — Churned:                   3 (0.3%)

## Output
- behavioral_segments.csv: 1,000 rows × 14 columns

In [2]:
import pandas as pd
import numpy as np

#load the labled dataset
df = pd.read_csv(r'F:\Projects\User Retention Study\User Churn Analysis\Data\cleaned\churn_labeled.csv')


#reconvert the date column
df['first_activity'] = pd.to_datetime(df['first_activity'])
df['last_activity'] = pd.to_datetime(df['last_activity'])

#check
print(df.shape)
print(df.columns.tolist())


(1000, 12)
['UserID', 'total_events', 'total_sessions', 'total_purchases', 'total_spend', 'unique_events', 'first_activity', 'last_activity', 'active_days', 'purchase_frequency', 'inactivity_days', 'churn_label']


In [3]:
#defining the activity level

median_events = df['total_events'].median()
median_purchases = df['total_purchases'].median()
median_spend = df['total_spend'].median()

print("Median total events:",median_events)
print("Median total purchases:",median_purchases)
print("Median total spend:",median_spend)



Median total events: 75.0
Median total purchases: 10.0
Median total spend: 2649.110063154056


In [4]:
#define activity level based on events and purchases

df['activity_level'] = df['total_events'].apply(
    lambda x: 'High Activity' if x >= median_events else 'Low Activity'
)

#check
print("Activity level Discription:")
print(df['activity_level'].value_counts())

Activity level Discription:
activity_level
High Activity    536
Low Activity     464
Name: count, dtype: int64


In [5]:
#combining  the activity level with churn label
df['segment'] = df['activity_level'] + '-' + df['churn_label']

#check 
print("Behavioral Segments:")
print(df['segment'].value_counts())

Behavioral Segments:
segment
High Activity-Active             499
Low Activity-Active              429
High Activity-Potential Churn     34
Low Activity-Potential Churn      32
Low Activity-Churned               3
High Activity-Churned              3
Name: count, dtype: int64


In [6]:
#Identify quite loyal users

#low activity + active + at least q purchase
quiet_loyal_mask = (
    (df['activity_level'] == 'Low Activity') &
    (df['churn_label'] == 'Active') &
    (df['purchase_frequency'] >= df['purchase_frequency'].median())
)

# Apply this special segment label 

df.loc[quiet_loyal_mask, 'segment'] = 'Low Activity - Retained (Quiet Loyal)' 

print("Update segment Distribution: ")
print(df['segment'].value_counts())

Update segment Distribution: 
segment
High Activity-Active                     499
Low Activity-Active                      243
Low Activity - Retained (Quiet Loyal)    186
High Activity-Potential Churn             34
Low Activity-Potential Churn              32
Low Activity-Churned                       3
High Activity-Churned                      3
Name: count, dtype: int64


In [7]:
#confirming all 1000 users
print("Total usera:", df['segment'].notna().sum())

#check any missing user for segment
print("Null segments:", df['segment'].isna().sum())

#percentage
print("\nSegment percentage breakdown:")
print((df['segment'].value_counts(normalize=True)*100).round(2))

Total usera: 1000
Null segments: 0

Segment percentage breakdown:
segment
High Activity-Active                     49.9
Low Activity-Active                      24.3
Low Activity - Retained (Quiet Loyal)    18.6
High Activity-Potential Churn             3.4
Low Activity-Potential Churn              3.2
Low Activity-Churned                      0.3
High Activity-Churned                     0.3
Name: proportion, dtype: float64


In [8]:
df.to_csv(r'F:\Projects\User Retention Study\User Churn Analysis\Data\cleaned\behavioral_segments.csv', index=False)

print(f"Final shape: {df.shape}")
print(f"Final segment distribution:")
print(df['segment'].value_counts())

Final shape: (1000, 14)
Final segment distribution:
segment
High Activity-Active                     499
Low Activity-Active                      243
Low Activity - Retained (Quiet Loyal)    186
High Activity-Potential Churn             34
Low Activity-Potential Churn              32
Low Activity-Churned                       3
High Activity-Churned                      3
Name: count, dtype: int64
